# Problem Set 5: Misspecification in a Rust-Style Replacement Model

# A Replacement Problem with Quadratic Maintenance Costs

A regional courier company operates a fleet of delivery vans. The fleet
manager indexes each van by its accumulated mileage
$x_t \in \{0, 1, \ldots, S-1\}$ (discretized into 5,000-mile bins) and
each period chooses between:

-   **Keep** ($d_t = 0$): pay maintenance cost $c(x_t, \theta)$.
-   **Replace** ($d_t = 1$): pay replacement cost $RC$ and reset mileage
    to zero.

The setup is the same as the Rust lecture in almost every respect:
Type-I Extreme Value choice shocks $\varepsilon_t(d)$, mileage
transitions with increments $\Delta x \in \{0, 1, 2\}$ governed by
$\theta_3$, and the Bellman / logit machinery from the lecture. The
**one** difference is the **maintenance cost function**.

Mechanical engineers at the courier company argue that wear on a
delivery van compounds: at low mileage maintenance costs are negligible,
but at high mileage they explode. They recommend a **pure quadratic**
specification:

$$
c(x, \theta_2) = \theta_2 \cdot x^2.
$$

This is the **true** data generating process (DGP) you will use to
simulate data. Then, you will play the role of an econometrician who
**does not know the truth** and estimates the model two ways:

1.  **Correctly specified** — quadratic cost, $c(x) = \theta_2 x^2$,
    estimating $(\theta_2, RC)$.
2.  **Misspecified** — linear cost, $c(x) = \theta_1 x$, estimating
    $(\theta_1, RC)$.

The goal is to see how well NFXP recovers the truth when the functional
form is right, and what goes wrong when it isn’t.

Throughout, use the following baseline parameters:

$$
\beta = 0.95, \quad S = 50, \quad RC = 20.0, \quad \theta_2 = 0.001078, \quad \theta_3 = (0.36,\, 0.48,\, 0.16).
$$

> **Calibration of $\theta_2$**
>
> The quadratic coefficient $\theta_2 = 0.001078$ is chosen so that the
> **best linear approximation** to the quadratic cost (OLS through the
> origin on the mileage grid $0, 1, \ldots, S-1$) has slope
> $\theta_1^{\text{lin}} = 0.04$. Exactly the linear cost coefficient
> used in the lecture. So the misspecified researcher is, in effect,
> fitting *the same linear cost the lecture used*, but to data generated
> by a quadratic DGP.

> **How this differs from the lecture**
>
> The lecture used a **linear** cost $c(x) = \theta_1 x$. In this
> problem set, the truth is **quadratic**: costs accelerate as the van
> ages, so the marginal cost of one more period of wear is larger at
> high mileage than at low mileage. Everything else is identical to the
> lecture.

# Building the Model

## Problem 1: The `VanModel` Struct

**(a)** Define a `VanModel` struct (using `@kwdef`) with the baseline
default values for $\beta$, $S$, $RC$, $\theta_2$, $\theta_3$, the
mileage grid $\bar{x} = (0, 1, \ldots, S-1)$, and a **pre-computed**
maintenance cost vector:

$$
c_i = \theta_2 \cdot \bar{x}_i^2.
$$

Follow the `RustModel` pattern from the lecture; the only structural
change is the cost vector.

In [1]:
# Your code here


**(b)** Create an instance `m = VanModel()` and plot $c(x)$ against
$\bar{x}$. On the same figure, also plot the **best linear fit**
$\theta_1^{\text{lin}} \cdot \bar{x}$ where $\theta_1^{\text{lin}}$ is
chosen by OLS through the origin. Verify that
$\theta_1^{\text{lin}} \approx 0.04$ (the lecture’s linear coefficient).
Where does the linear approximation under- and over-state the true cost?

> **Hint**
>
> OLS through the origin:
> $\hat\theta_1^{\text{lin}} = \frac{\sum_i \bar{x}_i \cdot c_i}{\sum_i \bar{x}_i^2}$.

In [1]:
# Your code here


**(c)** Bring in the `buildTransition(θ₃, S)` function from lecture and
use it to construct $F_0$ and $F_1$ for the baseline model.

In [1]:
# Your code here


# Solving the Dynamic Problem

## Problem 2: Value Function Iteration

The Bellman equation is **identical** to lecture — only the cost vector
inside differs:

$$
V(x) = \log\!\bigl(\exp(v^{\text{keep}}(x)) + \exp(v^{\text{replace}}(x))\bigr),
$$

$$
v^{\text{keep}} = -c + \beta \, F_0 \, V, \qquad v^{\text{replace}} = -RC \cdot \mathbf{1} + \beta \, F_1 \, V.
$$

**(a)** Bring in the lecture’s `iterateBellman` and `solveBellman`
functions, adapting them so they accept a `VanModel`. You should not
need to change the mathematical structure of the Bellman equation.

In [1]:
# Your code here


**(b)** Solve the baseline model. Plot $V(x)$ against $\bar{x}$ and
report the number of iterations to convergence.

In [1]:
# Your code here


**(c)** Write a function `solveVan(m::VanModel)` analogous to
`solveRust` from lecture that returns a named tuple
`(V, P, v_keep, v_replace)` where $P$ is the per-state probability of
replacement.

In [1]:
# Your code here


**(d)** Plot the replacement probability $P(\text{Replace} \mid x)$
against the mileage grid. Compare its shape — visually — to the
replacement probability from a model with *linear* cost. Specifically,
build a `RustModel`-equivalent with `θ₁ = 0.04` (the lecture default)
and overlay its $P(\text{Replace} \mid x)$ on the same figure. Where do
the two curves disagree most?

In [1]:
# Your code here


# Simulating Data

## Problem 3: Generating a Panel

**(a)** Adapt `simulateRust` from lecture into
`simulateVan(P_rep, θ₃, S, T; seed=42)`. The mileage transition rules
are identical to the lecture (mileage increments by
$\Delta \in \{0, 1, 2\}$; replacement resets to state 1, then
increments).

In [1]:
# Your code here


**(b)** Solve the baseline model and simulate $T = 50{,}000$ periods.
Report the **replacement rate** (fraction of periods with $d_t = 1$) and
the **mean mileage state**. Plot the mileage path for the first 300
periods; you should see the same sawtooth pattern as in lecture.

In [1]:
# Your code here


# NFXP — Correctly Specified

## Problem 4: Recovering the Quadratic Cost

You will now estimate $(RC, \theta_2)$ from the simulated data using
NFXP, knowing the **correct** functional form. As always, transition
probabilities $\theta_3$ are estimated separately in a first step (see
lecture); use $\hat\theta_3$ in the inner loop.

**(a)** Estimate $\hat{\theta}_3$ from `x_sim`, `d_sim` exactly as in
lecture. Confirm it is close to the true $(0.36, 0.48, 0.16)$.

In [1]:
# Your code here


**(b)** Write the NFXP objective
`nfxpObjectiveQuad(params, x_data, d_data, β, θ₃_hat, S)` that:

1.  Unpacks `params = [RC, θ₂]`.
2.  Returns a large penalty (e.g., `1e10`) if any parameter is negative.
3.  Builds a `VanModel` with the candidate parameters and solves the DP
    via `solveVan`.
4.  Returns the **negative** log-likelihood of the observed choices,
    using `max(P, 1e-15)` inside the `log` for numerical safety.

In [1]:
# Your code here


**(c)** Using `Optim.jl` with `NelderMead()` and the initial guess
`[15.0, 0.001]`, minimize the NFXP objective. Report
$(\hat{RC}, \hat{\theta}_2)$ alongside the true values in a small table.
The estimates should be close to the truth: confirming that **when the
functional form is correct, NFXP works as advertised.**

In [1]:
# Your code here


**(d)** Re-solve the model at the estimated parameters and plot
$\hat{P}(\text{Replace} \mid x)$ against the true
$P(\text{Replace} \mid x)$ from Problem 2(c). The two curves should
overlap almost perfectly.

In [1]:
# Your code here


# NFXP — Misspecified

## Problem 5: What Happens When We Force a Linear Cost

You will now play the role of an econometrician who (incorrectly)
believes the cost is linear: $c(x) = \theta_1 x$. The DGP is still
quadratic — the misspecification is purely on the researcher’s side.
Conveniently, this is *exactly* the model from lecture, so you can reuse
the lecture’s `RustModel` struct and `solveRust` function directly.

**(a)** Bring in (or copy) the `RustModel` struct and `solveRust`
function from the lecture. Then write a misspecified NFXP objective
`nfxpObjectiveLin(params, x_data, d_data, β, θ₃_hat, S)` that:

1.  Unpacks `params = [RC, θ₁]`.
2.  Builds a `RustModel` with the candidate $(RC, \theta_1)$ and solves
    the DP via `solveRust`.
3.  Otherwise has the same structure as `nfxpObjectiveQuad`.

In [1]:
# Your code here


**(b)** Minimize the misspecified objective with `NelderMead()` and
initial guess `[15.0, 0.03]`. Report
$(\hat{RC}^{\text{lin}}, \hat{\theta}_1^{\text{lin}})$ and compare to
the *true* $RC$ and the best fit linear coefficient from Problem 2(b).
Are the estimates biased? Why do you think that is?

In [1]:
# Your code here


## Problem 6: Comparing Implied Choice Probabilities

**(a)** Using the misspecified estimates, re-solve the linear-cost
`RustModel` to get the misspecified-implied replacement probability
$\hat{P}^{\text{lin}}(\text{Replace} \mid x)$. On a **single figure**,
plot three curves:

1.  The **true** $P(\text{Replace} \mid x)$ from Problem 2.
2.  The **correctly specified** estimate
    $\hat{P}^{\text{quad}}(\text{Replace} \mid x)$ from Problem 4(d).
3.  The **misspecified** estimate
    $\hat{P}^{\text{lin}}(\text{Replace} \mid x)$ from this problem.

In [1]:
# Your code here


**(b)** Describe in 2–3 sentences where the misspecified model fits well
and where it fails. In particular, comment on the shape of
$\hat{P}^{\text{lin}}$ around the **threshold mileage** (where
$P \approx 0.5$) versus the **tails** (very low and very high mileage).
Why does the linear specification systematically mismatch the curvature
of the true policy?

**(c)** **A counterfactual.** Suppose the firm is considering a **30%
subsidy on engine replacement**, lowering $RC$ to $0.7 \cdot RC$. Plot
the new replacement probabilities as a function of mileage under:

1.  The true parameters.
2.  The correctly specified estimates.
3.  The misspecified (linear) estimates.

Which estimator gives a counterfactual closer to the truth? Why?

> **The Lesson**
>
> Misspecification can leave in-sample fit looking “okay” while
> distorting the **shape** of the policy function. Because
> counterfactuals re-solve the model at a different $RC$, errors in the
> cost function’s shape get amplified: the misspecified model
> misforecasts the policy response.

In [1]:
# Your code here
